In [ ]:
# B11 VLM ROI Pilot & Hallucination Auditor (LOG #559)
# Uses Kaggle $10/day API quota. Runs on CPU with internet enabled.
import json, os, time, re, zipfile
from pathlib import Path
from PIL import Image
from kaggle_secrets import UserSecretsClient

# 1. Retrieve API Key
try:
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret("GEMINI_API_KEY")
except Exception:
    API_KEY = os.environ.get("GEMINI_API_KEY", "")

assert API_KEY, "Please add GEMINI_API_KEY to Kaggle Secrets (Add-ons -> Secrets)!"

# 2. Setup Gemini Client
import google.generativeai as genai
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel("gemini-1.5-flash")

# 3. Locate Dataset
DATA_DIR = Path("/kaggle/input/rsna-knee-b11-pilot-data")
IMG_DIR = Path("/kaggle/working/images")
IMG_DIR.mkdir(parents=True, exist_ok=True)

zip_path = DATA_DIR / "images.zip"
if zip_path.exists():
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(IMG_DIR)
elif (DATA_DIR / "images").exists():
    IMG_DIR = DATA_DIR / "images"

batch = json.load(open(DATA_DIR / "pilot_test_batch.json", encoding="utf-8"))
print(f"Loaded {len(batch)} pilot studies with 3-arm contrast sets.")

# 4. Helper function to call VLM
def call_vlm(img_path, prompt_text):
    try:
        img = Image.open(img_path)
        resp = model.generate_content([img, prompt_text])
        text = resp.text.strip()
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            return json.loads(m.group(0))
        return {"raw": text}
    except Exception as e:
        return {"error": str(e)}

# 5. Execute 3-Arm Benchmark
results = []
print("Starting 3-Arm Evaluation (60 API queries)...")

for i, case in enumerate(batch):
    uid = case["study_id"]
    stratum = case["stratum"]
    lat_gold = case["gold_lateral_oa"]
    med_gold = case["gold_medial_oa"]
    print(f"[{i+1}/{len(batch)}] Study {uid[:12]} ({stratum})...", flush=True)
    
    coronal_img = IMG_DIR / f"{uid}_coronal.png"
    blank_img = IMG_DIR / f"{uid}_blank.png"
    
    # Arm A: Real Slice + Report
    res_a = call_vlm(coronal_img, case["arm_a"]["prompt"])
    time.sleep(0.5)
    
    # Arm B: Real Slice Only
    res_b = call_vlm(coronal_img, case["arm_b"]["prompt"])
    time.sleep(0.5)
    
    # Arm C: Blank Slice + Report (Negative Hallucination Control)
    res_c = call_vlm(blank_img, case["arm_c"]["prompt"])
    time.sleep(0.5)
    
    results.append({
        "study_id": uid,
        "stratum": stratum,
        "gold_lateral_oa": lat_gold,
        "gold_medial_oa": med_gold,
        "arm_a": res_a,
        "arm_b": res_b,
        "arm_c": res_c,
    })

# 6. Audit & Scoring
print("\n" + "="*80)
print("=== B11 PILOT AUDIT RESULTS ===")
print("="*80)

a_hits, c_hits = 0, 0
a_fp, c_fp = 0, 0
total_pos = sum(1 for c in results if c["gold_lateral_oa"] == 1)
total_neg = sum(1 for c in results if c["gold_lateral_oa"] == 0)

for r in results:
    is_pos = (r["gold_lateral_oa"] == 1)
    a_pred = r["arm_a"].get("lateral_oa_present") is True
    c_pred = r["arm_c"].get("lateral_oa_present") is True
    
    if is_pos:
        if a_pred: a_hits += 1
        if c_pred: c_hits += 1
    else:
        if a_pred: a_fp += 1
        if c_pred: c_fp += 1

tpr_a = a_hits / max(1, total_pos)
tpr_c = c_hits / max(1, total_pos)
fpr_a = a_fp / max(1, total_neg)
fpr_c = c_fp / max(1, total_neg)
delta_gate = tpr_a - tpr_c

print(f"Total Positives: {total_pos} | Total Negatives: {total_neg}")
print(f"Arm A (Image + Report):  Sensitivity = {tpr_a:.1%} | False Positive Rate = {fpr_a:.1%}")
print(f"Arm C (Blank + Report):  Sensitivity = {tpr_c:.1%} (HALLUCINATION BASELINE) | False Positive Rate = {fpr_c:.1%}")
print(f"Separation Gap (Delta):  {delta_gate:+.1%} (Target Gate: >= +40.0%)")

if delta_gate >= 0.40:
    print("\n>>> VERDICT: GATE PASSED! Model uses true image pixels beyond hallucination! <<<")
else:
    print("\n>>> VERDICT: GATE FAILED / MARGINAL. High risk of report-conditioned hallucination. <<<")

with open("/kaggle/working/b11_pilot_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("Saved /kaggle/working/b11_pilot_results.json successfully!")
